# Level-Based Foraging — Custom Environment Example

Demonstrates:
1. Setting up the custom 10×10 foraging env with walls, traps, and fixed starting positions.
2. Random-agent rollout to validate the environment.
3. Partial observability mode.
4. IQL training using the shared `marl_utils` helper.

In [ ]:
import sys
from pathlib import Path

# Make project root importable
ROOT = Path().resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from discrete_action_space.lbf_grid import make_pz_env
from discrete_action_space.marl_utils import random_rollout, run_iql

## 1. Construct the environment

In [ ]:
# 3 agents, 10×10, fully observable (sight=10)
# Vertical wall at column 4 (rows 0–7), two traps, fixed starts
env = make_pz_env(
    players=3,
    field_size=(10, 10),
    sight=10,
    max_food=3,
    max_episode_steps=100,
    start_positions=[(0, 0), (0, 9), (9, 0)],
    wall_positions=[(r, 4) for r in range(8)],
    trap_positions=[(2, 2), (7, 7)],
    collision_penalty=-1.0,
    trap_penalty=-5.0,
)

print('Agents      :', env.possible_agents)
print('Obs space   :', env.observation_space(env.possible_agents[0]))
print('Action space:', env.action_space(env.possible_agents[0]))

## 2. Random-agent rollout (fully observable)

In [ ]:
print('=== Random rollout — fully observable ===')
results = random_rollout(env, n_episodes=3)
env.close()

## 3. Partial observability (sight = 3)

In [ ]:
env_partial = make_pz_env(
    players=3,
    field_size=(10, 10),
    sight=3,   # only see 3 cells in each direction
    max_food=3,
    start_positions=[(0, 0), (0, 9), (9, 0)],
)
print('Partial-obs obs space:', env_partial.observation_space(env_partial.possible_agents[0]))
print('=== Random rollout — partial observable ===')
random_rollout(env_partial, n_episodes=2)
env_partial.close()

## 4. IQL Training

Trains independent Q-learning agents using `marl_utils.run_iql`.

In [ ]:
def make_env():
    return make_pz_env(
        players=2,
        field_size=(10, 10),
        sight=10,
        max_food=3,
        max_episode_steps=100,
        start_positions=[(0, 0), (9, 9)],
        wall_positions=[(r, 4) for r in range(8)],
        trap_positions=[(2, 2), (7, 7)],
    )

results = run_iql(
    make_env_fn=make_env,
    n_frames=30_000,
    frames_per_batch=500,
    train_batch_size=64,
    save_folder='checkpoints/lbf_iql',
)
print('Training complete.')

## 5. Plot reward curve

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ep_rewards = results['episode_rewards']
fig, ax = plt.subplots(figsize=(10, 4))
for agent, rews in ep_rewards.items():
    window = 30
    rolling = np.convolve(rews, np.ones(window) / window, mode='valid')
    ax.plot(rolling, label=agent)
ax.set_xlabel('Episode')
ax.set_ylabel(f'Reward (rolling avg {window})')
ax.set_title('LBF IQL training')
ax.legend()
plt.tight_layout()
plt.savefig('checkpoints/lbf_iql/training_curve.png', dpi=120)
plt.show()